# SharePoint Verification
Confirm SharePoint connection has been configured and docs can be ingested. 

---


In [0]:
%run ../../notebooks/_resources/Utilities

## Unity Catalog Connection
Create a Unity Catalog connection to store your SharePoint credentials. 
https://docs.databricks.com/aws/en/ingestion/lakeflow-connect/sharepoint-connection
- **OAuth U2M: Databricks-managed** - The account used to authenticate is shared with all users.  Use a service account!
- **OAuth U2M: Custom-managed** - organization requires control over the OAuth application
- **OAuth M2M** - Use M2M for fully automated production pipelines that run without user interaction.

Create the connection manually through the Catalog Explorer UI, then update the connection_name in the Variable definitions cell, below.

In [0]:
dbutils.widgets.text("connector_name", "vantage_demo_sharepoint_connection", "SharePoint Connection Name")
dbutils.widgets.text("site_url", "", "SharePoint Site URL")

# Unity Catalog
connection_name = dbutils.widgets.get("connector_name")
sharepoint_site_url = dbutils.widgets.get("site_url")

# SharePoint
# sharepoint_site_url = "https://databricksinc.sharepoint.com/sites/VantageDemo/Shared%20Documents/Forms/AllItems.aspx"
# sharepoint_site_url = "https://bcgcloud.sharepoint.com/sites/Voyager/Shared%20Documents/Forms/AllItems.aspx" # Vantage


### Step 1 — Load raw PDFs from SharePoint

In [0]:
from pyspark.sql.functions import lit

# Read all PDF files from SharePoint as binary files

# This is a batch read. For automatic incremental ingestion, view the cells at the bottom of the notebook to see how to use this connector in Lakeflow Spark Declarative Pipelines
pdf_df = (spark.read
    .format("binaryFile") # Use this format for unstructured data
    .option("databricks.connection", connection_name) # User provides the name of their connection
    .option("recursiveFileLookup", True)
    .option("pathGlobFilter", "*.pdf")  # Only ingest PDF files
    .load(sharepoint_site_url)
    .select("*", "_metadata")
)

# Display the DataFrame to see the PDF files
display(pdf_df.limit(1))